# `torch.scatter_add` mimics mean aggregation

In [1]:
import dgl
import torch

Let's create a graph with source nodes probided in `src` and destination nodes in `dst` tensors, respecitvely. Each node has 2 features in `features` tensor

In [2]:
src = torch.tensor([0, 0, 1, 1])
dst = torch.tensor([1, 2, 0, 2])

features = torch.tensor([[1, 1], [2, 1], [1, 2]]).float()
features.requires_grad = True

what we want mean-aggregate features for each node based on its incoming edges

Baseline - do it with dgl:

In [3]:
graph = dgl.graph((src, dst))


target = dgl.ops.copy_u_mean(graph, features)

In [4]:
loss = target.sum()
loss.backward()

print("Result:")
print(target)
print("\nGradients:")
print(features.grad)


target_grad = features.grad

Result:
tensor([[2.0000, 1.0000],
        [1.0000, 1.0000],
        [1.5000, 1.0000]], grad_fn=<DivBackward0>)

Gradients:
tensor([[1.5000, 1.5000],
        [1.5000, 1.5000],
        [0.0000, 0.0000]])


In [5]:
del features

In [6]:
features = torch.tensor([[1, 1], [2, 1], [1, 2]]).float()


features.requires_grad = True

src_features = features[src]
dst_expanded = dst.unsqueeze(1).expand(-1, features.shape[1])

result_sum = torch.scatter_add(
    torch.zeros(features.shape[0], features.shape[1], dtype=features.dtype), 0, dst_expanded, src_features
)

counts = torch.scatter_add(
    torch.zeros(features.shape[0], dtype=torch.float), 0, dst, torch.ones_like(dst, dtype=torch.float)
)

result = result_sum / counts.unsqueeze(1).clamp(min=1)

loss = result.sum()
loss.backward()

print("Result:")
print(result)
print("\nGradients:")
print(features.grad)

torch.testing.assert_close(features.grad, target_grad)
torch.testing.assert_close(result, target)

Result:
tensor([[2.0000, 1.0000],
        [1.0000, 1.0000],
        [1.5000, 1.0000]], grad_fn=<DivBackward0>)

Gradients:
tensor([[1.5000, 1.5000],
        [1.5000, 1.5000],
        [0.0000, 0.0000]])


# Mimic edge softmax in raw torch

In [5]:
src = torch.tensor([0, 0, 1, 1])
dst = torch.tensor([1, 2, 0, 2])

edge_scores = torch.tensor([1.0, 2.0, 3.0, 4.0])
edge_scores.requires_grad_(True)

tensor([1., 2., 3., 4.], requires_grad=True)

In [6]:
graph = dgl.graph((src, dst))


target = dgl.nn.functional.edge_softmax(graph, edge_scores)
target

tensor([1.0000, 0.1192, 1.0000, 0.8808], grad_fn=<EdgeSoftmaxBackward>)

In [7]:
target.sum().backward()
target_grad = edge_scores.grad
target_grad

tensor([0.0000e+00, 7.4506e-09, 0.0000e+00, 5.9605e-08])

In [8]:
del edge_scores

In [9]:
edge_scores = torch.tensor([1.0, 2.0, 3.0, 4.0])
edge_scores.requires_grad_(True)

# normalize over incoming edges to each destination
exp_scores = torch.exp(edge_scores)

# sum-exp scores per DESTINATION node like in default DGL's setup
num_nodes = max(src.max().item(), dst.max().item()) + 1
exp_sum = torch.scatter_add(torch.zeros(num_nodes, dtype=edge_scores.dtype), 0, dst, exp_scores)

# gather denominators for each edge
denominators = exp_sum[dst]

# normalize
edge_softmax = exp_scores / denominators

edge_softmax.sum().backward()


torch.testing.assert_close(edge_softmax, target)
torch.testing.assert_close(target_grad, edge_scores.grad)